In [1]:
!pip install 'powerlaw'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 kB 39.2 MB/s eta 0:00:00


In [2]:
import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, get_market_tables, save_spark_dataframe, load_spark_dataframe

b2.get_file('modules/metaorder.py')
b2.get_file('modules/LMF.py')
b2.get_file('modules/calibration.py')

import metaorder as metaorder
import LMF as LMF
import calibration as calibration

import random
import math
import pandas as pd
import polars as pl
import numpy as np
from pandas import StringDtype
from datetime import timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import LogFormatterSciNotation

from statsmodels.sandbox.stats.runs import runstest_1samp
import powerlaw
import itertools
import pylab
import scipy.stats
import scipy.fft
from scipy.optimize import curve_fit
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.stattools import acf
from scipy.stats import t

from typing import Literal

# Synthetic metaorder configuration settings

## Fixed settings

The following settings define the market and trader participation assumptions used throughout the study and should not be modified.

- **trader_participation_distribution**: Distribution of trader participation rates. Fixed as `'power'`, corresponding to a power-law distribution.
- **exchange**: Exchange from which the data were obtained. Fixed as `'XJSE'`.

## User-configurable settings

The following settings control the construction of synthetic metaorders.

- **config**: Objective function used to select the optimal synthetic metaorder configuration.
    - `'metaorder'`: minimises \( e_M \)
    - `'LMF'`: minimises \( e_{LMF} \)
    - `'combined'`: minimises \( e_{combo} \)
- **grouped**: If `True`, the parameters \(N\) and \(\delta\) are constrained to be identical across all years. If `False`, they are estimated separately for each year.

In [3]:
trader_participation_distribution = 'power'
exchange                          = 'XJSE'
config                            = 'combined'
estimation_method                 = 'PSD'
grouped                           = False

In [4]:
%%time
files = b2.list_files(path = f'data/grid_search_9/{exchange}/{trader_participation_distribution}')

for f in files:
    b2.get_file(f'data/grid_search_9/{exchange}/{trader_participation_distribution}/{f}')

CPU times: user 2.96 s, sys: 154 ms, total: 3.11 s
Wall time: 36.8 s


In [5]:
data = pl.scan_parquet(f'*{trader_participation_distribution}_grid_search.parquet').collect()

if trader_participation_distribution == 'homogenous':
    data.sort(['ticker', 'year', 'N'])
    data.head(10)
if trader_participation_distribution == 'power':
    data.sort(['ticker', 'year', 'N', 'delta'])
    data.head(10)

In [8]:
b2.get_file('top_250.csv')
top_250 = pd.read_csv('top_250.csv')

top_250_tickers = top_250['Ticker'].tolist()

In [9]:
%%time
files = b2.list_files(path = 'data/trades/top_250')

for f in files:
    b2.get_file(f'data/trades/top_250/{f}')

CPU times: user 8.71 s, sys: 4.45 s, total: 13.2 s
Wall time: 58.2 s


In [13]:
def load_stock(parquet):
    try:
        df = pl.read_parquet(parquet)
        return df.sort(['DateTime', 'ExchangeSequenceNo'])
    except FileNotFoundError:
        return None

stocks = {
    ticker: df
    for ticker in top_250_tickers
    if (df := load_stock(f'{ticker}_{exchange}.parquet')) is not None
}

In [24]:
b2.create_folder('data', ensure_exists = True)
b2.create_folder('data/metaorders', ensure_exists = True)
b2.create_folder(f'data/metaorders/{exchange}', ensure_exists = True)
b2.create_folder(f'data/metaorders/{exchange}/calibrated', ensure_exists = True)
b2.create_folder(f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}', ensure_exists = True)
b2.create_folder(f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}/{config}', ensure_exists = True)

In [25]:
%%time
configs                           = calibration.find_configs(data = data, grouped = grouped,
                                                 trader_participation_distribution = 'power',
                                                 config = config, estimation_method = estimation_method, 
                                                 lambda_penalty = 1, mix = [1, 1, 1])

if (grouped == True):
    list_cols = [
        col.name
            for col in configs
                if col.dtype.base_type() == pl.List
    ]
    configs = configs.explode(list_cols)

if (grouped == True):
    flag = 'grouped'
else:
    flag = 'ungrouped'

counter    = 1
start_from = 'BHG'
start      = False
for ticker, stock_data in stocks.items():

    if ticker == start_from:
        start = True

    if not start:
        continue
        
    print(f'{counter}: {ticker}')
    counter += 1

    optimal = configs.filter(pl.col('ticker') == ticker)
    if optimal.is_empty():
        continue

    if trader_participation_distribution == 'homogenous':
        N     = optimal['N'].to_list()
        group = 'year'
        
        if len(N) < 3:
            continue

        stock_features = metaorder.metaorder_data(stock_data, N = N, group = group, trader_participation_distribution = trader_participation_distribution)
        
    if trader_participation_distribution == 'power':
        N     = optimal['N'].to_list()
        delta = optimal[f'$\\delta$'].to_list()
        group = 'year'
        
        if len(N) < 3:
            continue

        stock_features = metaorder.metaorder_data(stock_data, N = N, group = group, trader_participation_distribution = trader_participation_distribution, delta = delta)

    if not stock_features.is_empty():

        stock_features.write_parquet(f'{ticker}_{exchange}_{trader_participation_distribution}_{config}_{flag}_metaorder_data_calibrated.parquet')
        b2.put_file(f'{ticker}_{exchange}_{trader_participation_distribution}_{config}_{flag}_metaorder_data_calibrated.parquet',
                    f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}/{config}')


1: BHG
2: BTI
3: ANH
4: PRX
5: CFR
6: GLN
7: AGL
8: ANG
9: NPN
10: GFI
11: SBK
12: FSR
13: CPI
14: VAL
15: MTN
16: VOD
17: IMP
18: S32
19: ABG
20: SLM
21: DSY
22: HAR
23: SHP
24: SSW
25: NPH
26: BID
27: NED
28: SOL
29: OUT
30: RNI
31: KIO
32: NRP
33: REM
34: INP
35: PPH
36: MNP
37: BVT
38: PAN
39: EXX
40: CLS
41: OMU
42: CCO
43: APN
44: GRT
45: QLT
46: WHL
47: TBS
48: MTM
49: SNT
50: ARI
51: MRP
52: DRD
53: RDF
54: HMN
55: INL
56: AVI
57: SRE
58: N91
59: BOX
60: KST
61: TKG
62: VKE
63: RES
64: DCP
65: FFB
66: KRO
67: OPA
68: TFG
69: SRI
70: PMR
71: TGA
72: HYP
73: TRU
74: ADH
75: MTH
76: APH
77: NTC
78: WBC
79: CML
80: LHC
81: NY1
82: DTC
83: LTE
84: BYI
85: GTC
86: EQU
87: MSP
88: OMN
89: JSE
90: PIK
91: HCI
92: ATT
93: GND
94: SPP
95: ITE
96: AFE
97: RLO
98: WBO
99: ARL
100: ISO
101: SUI
102: SAP
103: CVW
104: AFH
105: SDO
106: PPC
107: SAC
108: SSS
109: RBX
110: BLU
111: AEL
112: BAT
113: RCL
114: THA
115: CLI
116: IPF
117: HIL
118: HET
119: OCE
120: TSG
121: CAA
122: DIB
123: RFG
1

In [29]:
N_grid     = [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500]
N_to_index = {n: i + 1 for i, n in enumerate(N_grid)}

configs = ['metaorder', 'LMF', 'combined']

results = []
for objective in configs:
    
    configs = calibration.find_configs(data = data, grouped = grouped,
                                   trader_participation_distribution = 'power',
                                   config = objective, estimation_method = estimation_method, 
                                   lambda_penalty = 1, mix = [1, 1, 1])
    
    df = configs.with_columns(pl.col('N').replace_strict(N_to_index).alias('N_index'))

    variances_table    = df.group_by('ticker').agg(pl.col('N_index').var().alias('N index variance'), pl.col(f'$\\delta$').var().alias(f'$\\delta$ variance'))

    results.append({'Objective': objective, 
                    'Average variance (N_index)': variances_table['N index variance'].mean(),
                    f'Average variance ($\\delta$)': variances_table[f'$\\delta$ variance'].mean()})

variance_table = pl.DataFrame(results)
print(variance_table.to_pandas().to_latex(index = False, float_format = '%.2f'))

\begin{tabular}{lrr}
\toprule
Objective & Average variance (N_index) & Average variance ($\delta$) \\
\midrule
metaorder & 5.82 & 1.25 \\
LMF & 4.90 & 1.03 \\
combined & 4.22 & 1.00 \\
\bottomrule
\end{tabular}

